In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("sample_ecommerce_dirty.csv")

In [3]:
df.head()

,OrderID,OrderDate,CustomerName,ProductName,Category,City,Quantity,UnitPrice
0,ORD1001,2026-09-01,Asha Rao,Wireless Mouse,Accessories,Hyderabad,2.0,799
1,ORD1002,2026-09-01,Rohan Das,Mechanical Keyboard,Accessories,Bengaluru,1.0,2499
2,ORD1003,2026-09-02,Meera Shah,USB-C Hub,Electronics,Mumbai,3.0,1899
3,ORD1004,2026-09-02,Kabir Sen,Laptop Stand,Accessories,hyderabad,2.0,1299
4,ORD1005,2026-09-03,Tara Iyer,Noise-Cancelling Headphones,Audio,Chennai,1.0,6499


In [6]:
print("Shape:", df.shape)

Shape: (31, 8)


In [7]:
print("Columns:", df.columns.tolist())

Columns: ['OrderID', 'OrderDate', 'CustomerName', 'ProductName', 'Category', 'City', 'Quantity', 'UnitPrice']


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   OrderID       31 non-null     object 
 1   OrderDate     31 non-null     object 
 2   CustomerName  31 non-null     object 
 3   ProductName   31 non-null     object 
 4   Category      31 non-null     object 
 5   City          31 non-null     object 
 6   Quantity      30 non-null     float64
 7   UnitPrice     31 non-null     object 
dtypes: float64(1), object(7)
memory usage: 2.1+ KB


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   OrderID       31 non-null     object 
 1   OrderDate     31 non-null     object 
 2   CustomerName  31 non-null     object 
 3   ProductName   31 non-null     object 
 4   Category      31 non-null     object 
 5   City          31 non-null     object 
 6   Quantity      30 non-null     float64
 7   UnitPrice     31 non-null     object 
dtypes: float64(1), object(7)
memory usage: 2.1+ KB


In [10]:
df.describe(include="all")

,OrderID,OrderDate,CustomerName,ProductName,Category,City,Quantity,UnitPrice
count,31,31,31,31,31,31,30.000000,31
unique,30,15,30,14,6,9,NaN,16
top,ORD1002,2026-09-01,Rohan Das,Mechanical Keyboard,Electronics,Chennai,NaN,2499
freq,2,3,2,3,9,6,NaN,3
mean,NaN,NaN,NaN,NaN,NaN,NaN,1.866667,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,1.074255,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,NaN


In [12]:
print("Missing values by column:")
df.isnull().sum()


Missing values by column:


OrderID         0
OrderDate       0
CustomerName    0
ProductName     0
Category        0
City            0
Quantity        1
UnitPrice       0
dtype: int64

In [13]:
print("Missing percentage by column:")
print((df.isnull().sum() / len(df) * 100).round(2))


Missing percentage by column:
OrderID         0.00
OrderDate       0.00
CustomerName    0.00
ProductName     0.00
Category        0.00
City            0.00
Quantity        3.23
UnitPrice       0.00
dtype: float64


In [14]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated OrderID values:", df["OrderID"].duplicated().sum())

Exact duplicate rows: 1
Repeated OrderID values: 1


In [15]:
df[df["OrderID"].duplicated(keep=False)].sort_values("OrderID")

,OrderID,OrderDate,CustomerName,ProductName,Category,City,Quantity,UnitPrice
1,ORD1002,2026-09-01,Rohan Das,Mechanical Keyboard,Accessories,Bengaluru,1.0,2499
30,ORD1002,2026-09-01,Rohan Das,Mechanical Keyboard,Accessories,Bengaluru,1.0,2499


In [17]:
df = df.drop_duplicates().copy()
print("Shape after removing exact duplicate rows:", df.shape)

Shape after removing exact duplicate rows: (30, 8)


In [18]:
for col in ["CustomerName", "ProductName", "Category"]:
    df[col] = df[col].astype("string").str.strip().str.title()

In [19]:
city_map = {
    "hyderabad": "Hyderabad",
    "bengaluru": "Bengaluru",
    "mumbai": "Mumbai",
    "chennai": "Chennai",
    "delhi": "Delhi",
    "pune": "Pune"
}
city_original = df["City"].astype("string").str.strip()
df["City"] = city_original.str.lower().map(city_map).fillna(city_original.str.title())

In [20]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

In [21]:
df["UnitPrice"] = pd.to_numeric(
    df["UnitPrice"].astype("string").str.replace(",", "", regex=False),
    errors="coerce"
)

print(df.dtypes)
print("Missing values after conversion:")
print(df.isnull().sum())

OrderID                 object
OrderDate       datetime64[ns]
CustomerName    string[python]
ProductName     string[python]
Category        string[python]
City                    object
Quantity               float64
UnitPrice                Int64
dtype: object
Missing values after conversion:
OrderID         0
OrderDate       0
CustomerName    0
ProductName     0
Category        0
City            0
Quantity        1
UnitPrice       0
dtype: int64


In [22]:
invalid_quantity = df["Quantity"].isna() | (df["Quantity"] <= 0)
invalid_price = df["UnitPrice"].isna() | (df["UnitPrice"] <= 0)
invalid_date = df["OrderDate"].isna()
invalid_city = df["City"].isna() | df["City"].astype("string").str.strip().eq("")

In [23]:
print("Rows with missing/non-positive quantity:", int(invalid_quantity.sum()))
print("Rows with missing/non-positive price:", int(invalid_price.sum()))
print("Rows with invalid date:", int(invalid_date.sum()))
print("Rows with missing city:", int(invalid_city.sum()))


Rows with missing/non-positive quantity: 1
Rows with missing/non-positive price: 1
Rows with invalid date: 0
Rows with missing city: 0


In [26]:
print("Invalid quantity rows:")
print(df.loc[invalid_quantity, ["OrderID", "ProductName", "Quantity"]])
print("Invalid price rows:")
print(df.loc[invalid_price, ["OrderID", "ProductName", "UnitPrice"]])

Invalid quantity rows:
    OrderID ProductName  Quantity
28  ORD1029   Usb-C Hub       NaN
Invalid price rows:
    OrderID ProductName  UnitPrice
29  ORD1030  Smart Plug       -699


In [27]:
needs_review = df.loc[invalid_quantity | invalid_price | invalid_date | invalid_city].copy()
needs_review.to_csv("ecommerce_rows_requiring_review.csv", index=False)

In [28]:
needs_review = df.loc[invalid_quantity | invalid_price | invalid_date | invalid_city].copy()
needs_review.to_csv("ecommerce_rows_requiring_review.csv", index=False)

In [29]:
valid_mask = ~(invalid_quantity | invalid_price | invalid_date | invalid_city)
clean = df.loc[valid_mask].copy()
print("Rows retained for analysis:", len(clean))

Rows retained for analysis: 28


In [30]:
highest_price_index = clean["UnitPrice"].idxmax()
highest_priced_product = clean.loc[
    highest_price_index, ["OrderID", "ProductName", "UnitPrice", "City"]
]
print(highest_priced_product)


OrderID                 ORD1019
ProductName    Portable Monitor
UnitPrice                 12999
City                    Chennai
Name: 18, dtype: object


In [33]:
clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]

city_revenue = (
    clean.groupby("City", as_index=False)["Revenue"]
         .sum()
         .sort_values("Revenue", ascending=False)
)
print(city_revenue)
print("Total revenue:", clean["Revenue"].sum())
print("Highest-revenue city:", city_revenue.iloc[0]["City"])


        City  Revenue
1    Chennai  44992.0
4     Mumbai  37689.0
0  Bengaluru  30092.0
2      Delhi  26891.0
3  Hyderabad  21794.0
5       Pune  17489.0
Total revenue: 178947.0
Highest-revenue city: Chennai


In [ ]:
issue_log = pd.DataFrame([
    {
        "Issue": "Missing quantity",
        "Possible cause": "Incomplete form entry or failed import",
        "Recommended solution": "Check the source order; correct from the system of record. Exclude from revenue totals until resolved."
    },
    {
        "Issue": "Exact duplicate row",
        "Possible cause": "Repeated upload or retry",
        "Recommended solution": "Remove exact duplicates; use transaction/line-item identifiers to distinguish legitimate items in the same order."
    },
    {
        "Issue": "Comma-formatted price",
        "Possible cause": "Currency formatting included in a numeric field",
        "Recommended solution": "Remove approved thousands separators and convert with errors='coerce'; validate currency and units."
    },
    {
        "Issue": "Negative unit price",
        "Possible cause": "Data entry error or a return/credit represented in the wrong field",
        "Recommended solution": "Check the source. Do not silently treat it as a valid product price; keep it for review."
    },
    {
        "Issue": "Inconsistent city names",
        "Possible cause": "Different capitalization or leading/trailing spaces",
        "Recommended solution": "Trim spaces and map known variants to a canonical city name."
    }
])
print(issue_log.to_string(index=False))
